# Lab21 — Khôi phục checkpoint và đo bổ sung
**Nguyễn Nhân Sâm · 2A202602672.** Chọn runtime **T4**, chạy các ô theo thứ tự.

Upload ZIP `lab21_2A202602672_core_results_20261007T124407Z.zip` (~478 MB).
Notebook khôi phục nguồn và adapter đã chạy; không chạy lại core 1–7.
Các phép đo mới: 15 câu regression, merge/hot-swap 2 adapter, rank sweep nếu còn giờ.
Mỗi bước tự tải ZIP bằng chứng nhỏ; giữ các ZIP đó và gửi lại để cập nhật báo cáo.

**Chưa có kết quả GPU của notebook này.** Baseline/verdict cũ giữ nguyên.
Lần đo mới có thể khác vì revision model/tokenizer của lần cũ không được ghi.
Ca thua regression không được gọi thành ca thua target. Không tự nhận điểm thưởng khi chưa đủ bằng chứng.
Hạn bắt đầu bước mới: **23:20 ngày 07/10/2026**, dành 10 phút tổng hợp.
Lệnh đang chạy có thể vượt mốc; guard chỉ kiểm tra thời gian trước khi bắt đầu.

In [ ]:
import pathlib, subprocess, sys, os, json, hashlib, datetime, zipfile, io, urllib.request
ROOT=pathlib.Path('/content/Lab21-deadline-followup')
SOURCE='90679f91842e9f42c9192b9c5eab316935705e9c'
if not ROOT.exists():
    subprocess.run(['git','clone','--no-checkout','https://github.com/Nguyen-Sam-sheep-zzz/Day21-Track3-Finetuning-Lab.git',str(ROOT)],check=True)
    subprocess.run(['git','checkout','--detach',SOURCE],cwd=ROOT,check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip()==SOURCE
os.chdir(ROOT)
os.environ.update(COMPUTE_TIER='T4',EPOCHS='2',MASK_MODE='assistant-only')
os.environ.pop('BASE_MODEL',None); os.environ.pop('EVAL_LIMIT',None)
import torch
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0), 'Cần runtime T4.'
print('GPU',torch.cuda.get_device_name(0),'torch',torch.__version__)

In [ ]:
from google.colab import files
EXPECTED_NAME='lab21_2A202602672_core_results_20261007T124407Z.zip'
EXPECTED_SHA='78f715550078cf33e3906e3df50ce23164ee259182fb29b2b83887519a407b1a'
uploaded=files.upload()
assert set(uploaded)=={EXPECTED_NAME}, 'Chỉ chọn đúng ZIP core_results gốc.'
raw=uploaded[EXPECTED_NAME]
assert hashlib.sha256(raw).hexdigest()==EXPECTED_SHA, 'ZIP khác checkpoint đã kiểm tra.'
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert len(set(z.namelist()))==len(z.namelist())
    for info in z.infolist():
        p=pathlib.PurePosixPath(info.filename)
        assert not p.is_absolute() and '..' not in p.parts and '\\' not in info.filename
        assert info.filename.startswith('lab21_2A202602672/')
        assert ((info.external_attr >> 16) & 0o170000) != 0o120000, 'Không nhận symlink.'
        if info.is_dir(): continue
        rel=info.filename.removeprefix('lab21_2A202602672/')
        if not rel.startswith(('results/','adapters/correct/','adapters/attn_only/')): continue
        dest=ROOT/rel; content=z.read(info)
        if dest.exists() and dest.read_bytes()!=content:
            raise RuntimeError('Runtime có artifact khác: '+rel+'. Dùng runtime mới để giữ bằng chứng.')
        dest.parent.mkdir(parents=True,exist_ok=True); dest.write_bytes(content)
del uploaded,raw
print('Đã kiểm tra SHA256 và khôi phục 2 adapter + results.')

In [ ]:
manifest=json.loads((ROOT/'results/experiment_manifest.json').read_text(encoding='utf-8'))
versions=manifest['environment']
packages=['transformers','tokenizers','peft','trl','accelerate','datasets','bitsandbytes','safetensors']
pins=[name+'=='+versions[name] for name in packages]
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt',*pins],check=True)
sys.path.insert(0,str(ROOT/'src'))
from labkit import data,evidence,config
seed_rows=[json.loads(s) for s in (ROOT/'data/train_seed.jsonl').read_text(encoding='utf-8').splitlines() if s.strip()]
train,val=data.split(seed_rows,train_frac=.9,seed=42)
(ROOT/'data/split').mkdir(exist_ok=True)
for name,rows in [('train',train),('val',val)]:
    (ROOT/f'data/split/{name}.jsonl').write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in rows),encoding='utf-8')
for name,expected in manifest['context']['file_sha256'].items():
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest()==expected, 'Dataset/split khác checkpoint: '+name
print('Nguồn, split seed42 và checksum eval đúng; version mới sẽ được ghi vào bằng chứng bổ sung.')

In [ ]:
HELPER_COMMIT='384a8aeee7831ff8c68bf0fec1cb55205df08a71'
HELPER_SHA='594267585022da4be5d002af50301cad49043ab388e197a99a018c8dbdcc8bef'
url='https://raw.githubusercontent.com/Nguyen-Sam-sheep-zzz/Day21-Track3-Finetuning-Lab/'+HELPER_COMMIT+'/student_tools/deadline_experiments.py'
helper=urllib.request.urlopen(url).read()
assert hashlib.sha256(helper).hexdigest()==HELPER_SHA, 'Helper khác bản đã kiểm tra.'
(ROOT/'student_tools').mkdir(exist_ok=True)
(ROOT/'student_tools/deadline_experiments.py').write_bytes(helper)
(ROOT/'followup_helper_receipt.json').write_text(json.dumps({'helper_commit':HELPER_COMMIT,'helper_sha256':HELPER_SHA,'core_source_commit':SOURCE},indent=2),encoding='utf-8')
print('Helper cố định:',HELPER_COMMIT)

In [ ]:
STOP_AT='2026-10-07T16:20:00Z'
def download_evidence(stage):
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    dest=ROOT/f'lab21_2A202602672_followup_{stage}_{stamp}.zip'
    with zipfile.ZipFile(dest,'w',zipfile.ZIP_DEFLATED) as z:
        for folder in ['bonus_results','bonus_logs']:
            for p in sorted((ROOT/folder).rglob('*')):
                if p.is_file(): z.write(p,'lab21_2A202602672/'+str(p.relative_to(ROOT)))
        z.write(ROOT/'followup_helper_receipt.json','lab21_2A202602672/followup_helper_receipt.json')
    print('Evidence ZIP:',dest.name,'SHA256:',hashlib.sha256(dest.read_bytes()).hexdigest())
    files.download(str(dest))

def run_stage(stage):
    (ROOT/'bonus_logs').mkdir(exist_ok=True)
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    log=ROOT/f'bonus_logs/{stage}_{stamp}.txt'
    cmd=[sys.executable,'-u','student_tools/deadline_experiments.py','--stages',stage,'--stop-at-utc',STOP_AT]
    try:
        with log.open('w',encoding='utf-8') as f:
            p=subprocess.Popen(cmd,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
            for line in p.stdout:
                print(line,end=''); f.write(line); f.flush()
            rc=p.wait()
        print('Stage',stage,'exit',rc)
    finally:
        download_evidence(stage)
    if rc not in (0,2): raise RuntimeError('Stage lỗi; đã tải log. Giữ ZIP để kiểm tra.')

## 1. Đo regression trước — ưu tiên bằng chứng cho mục 3.4
Không huấn luyện lại. Phép đo 15 câu và so với raw baseline đã đóng băng.

In [ ]:
run_stage('regression')

## 2. B1 — merge và đổi 2 adapter trên cùng base
Đo đủ 50 câu trước/sau merge; đo correct và attn_only trên cùng một model base.

In [ ]:
run_stage('b1')

## 3. B4 — rank sweep nếu còn thời gian
r16 dùng adapter cũ nhưng đánh giá lại; r8/r64 huấn luyện bổ sung cùng recipe.
Mỗi rank mới cần ít nhất 15 phút dự phòng khi bắt đầu. Thiếu giờ sẽ ghi partial/incomplete.

In [ ]:
run_stage('b4')

## Kết thúc
Giữ cả ba ZIP followup đã tải. **Không xóa bốn ZIP gốc và không chạy lại core.**
Các file JSON/log của lần đo mới phải được kiểm tra trước khi thêm vào report.
Phản tư cá nhân vẫn cần Sâm đọc và chỉnh theo trải nghiệm thực tế.